In [1]:
import scanpy as scp
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns
import numpy as np
import pandas as pd
import io
import os

from parameters import *

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [2]:
adata = scp.read("results/QC.h5ad")

# Clustering

In [32]:
fig,ax = plt.subplots(6,2,figsize=[20,60])
fig2,ax2 = plt.subplots(6,1,figsize=[10,40])
fig3,ax3 = plt.subplots(6,1,figsize=[10,40])

pca = "X_pca"
umap = "X_umap"

#Clustering global
scp.tl.leiden(adata,resolution=RESOLUTION, key_added="leiden_global")
    
    #Scatter plots
sns.scatterplot(x=adata.obsm[umap][:,0],y=adata.obsm[umap][:,1],hue=adata.obs["leiden_global"],ax=ax[0,0])
ax[0,0].set_title("global",fontsize=30)   
ax[0,0].legend(bbox_to_anchor=(1,1), loc="upper left")
for cluster in adata.obs["leiden_global"].unique():
    x = adata[adata.obs["leiden_global"]==cluster,:].obsm[umap][:,0].mean()
    y = adata[adata.obs["leiden_global"]==cluster,:].obsm[umap][:,1].mean()
    ax[0,0].text(x,y,cluster,backgroundcolor="lightgrey")
    
    #Make cluster contributions by Condition
data = adata.obs.groupby(by=["leiden_global","Condition"]).count()
dCum = adata.obs.groupby(by=["leiden_global"]).count()
color = {"whole":"grey","kitScal":"blue","CD41":"red","CD45":"red"}
for j,condition in enumerate(data.index.unique(level=1)):
        
    d = data.xs(condition,level=1)
    
    sns.barplot(x=d.index,y=dCum["Cell"],color=color[condition],ax=ax2[0])

    dCum.loc[:,"Cell"] -= d.loc[:,"Cell"]
                
ax2[0].set_title("Global",fontsize=30)   
ax2[0].legend(handles=[mlines.Line2D([],[],color=color[j],label=j) for j in data.index.unique(level=1)],fontsize=30)  

    #Make Condition contributions by cluster
dCum = adata.obs.groupby(by=["Condition"]).count()
color = {"0":"blue","1":"orange","2":"green","3":"red","4":"purple","5":"brown","6":"pink","7":"grey","8":"lightblue","9":"darkorange","10":"lightgreen"}
for j,condition in enumerate(data.index.unique(level=0)):
        
    d = data.xs(condition,level=0)
    
    sns.barplot(x=d.index,y=dCum["Cell"],color=color[condition],ax=ax3[0])

    dCum.loc[:,"Cell"] -= d.loc[:,"Cell"]
                
ax3[0].set_title("Global",fontsize=30)   
ax3[0].legend(handles=[mlines.Line2D([],[],color=color[j],label=j) for j in data.index.unique(level=0)],fontsize=20)  

#Make clusters by time
for i,time in enumerate(np.sort(adata.obs["Time"].unique())):
    adataAux = scp.read("results/dimensional_reduction_"+time+".h5ad")
    
    scp.pp.neighbors(adataAux,use_rep=pca,metric=METRIC,n_pcs=PCA,knn=KNN)
    scp.tl.leiden(adataAux,resolution=RESOLUTION)
    
    #Add to main data
    adata.obs.loc[adataAux.obs.index,"leiden_by_times"] = [time+"_"+j for j in adataAux.obs.loc[:,"leiden"]]
    
    X = adata[adataAux.obs.index].obsm["X_umap"]
    
    #Scatter plots
    #Own umap
    sns.scatterplot(x=adataAux.obsm[umap][:,0],y=adataAux.obsm[umap][:,1],hue=adataAux.obs["leiden"],ax=ax[i+1,1])
    ax[i+1,1].set_title(time+" leiden",fontsize=30)
    ax[i+1,1].legend(bbox_to_anchor=(1,1), loc="upper left")
    for louvain in adataAux.obs["leiden"].unique():
        x = adataAux[adataAux.obs["leiden"]==louvain,:].obsm[umap][:,0].mean()
        y = adataAux[adataAux.obs["leiden"]==louvain,:].obsm[umap][:,1].mean()
        ax[i+1,1].text(x,y,louvain,backgroundcolor="lightgrey")

    #Global umap
    sns.scatterplot(x=adata.obsm["X_umap"][:,0],y=adata.obsm["X_umap"][:,1],color="lightgrey",ax=ax[i+1,0])
    sns.scatterplot(x=X[:,0],y=X[:,1],hue=adataAux.obs["leiden"],ax=ax[i+1,0])
    ax[i+1,0].set_title(time+" leiden",fontsize=30)
    ax[i+1,0].legend(bbox_to_anchor=(1,1), loc="upper left")
    ax[i+1,0].axis([ax[0,0].get_xlim()[0],ax[0,0].get_xlim()[1],ax[0,0].get_ylim()[0],ax[0,0].get_ylim()[1]])
    for louvain in adataAux.obs["leiden"].unique():
        x = X[adataAux.obs["leiden"]==louvain,0].mean()
        y = X[adataAux.obs["leiden"]==louvain,1].mean()
        ax[i+1,0].text(x,y,louvain,backgroundcolor="lightgrey")
        
    #Make cluster contributions by Condition
    data = adataAux.obs.groupby(by=["leiden","Condition"]).count()
    dCum = adataAux.obs.groupby(by=["leiden"]).count()
    color = {"whole":"grey","kitScal":"blue","CD41":"red","CD45":"red"}
    for j,condition in enumerate(data.index.unique(level=1)):
        
        d = data.xs(condition,level=1)
    
        sns.barplot(x=d.index,y=dCum["Cell"],color=color[condition],ax=ax2[i+1])

        dCum.loc[:,"Cell"] -= d.loc[:,"Cell"]
                
    ax2[i+1].set_title(time,fontsize=30)   
    ax2[i+1].legend(handles=[mlines.Line2D([],[],color=color[j],label=j) for j in data.index.unique(level=1)],fontsize=30)  

    #Make Condition contributions by cluster
    dCum = adataAux.obs.groupby(by=["Condition"]).count()
    color = {"0":"blue","1":"orange","2":"green","3":"red","4":"purple","5":"brown","6":"pink","7":"grey"}
    for j,condition in enumerate(data.index.unique(level=0)):
        
        d = data.xs(condition,level=0)
    
        sns.barplot(x=d.index,y=dCum["Cell"],color=color[condition],ax=ax3[i])

        dCum.loc[:,"Cell"] -= d.loc[:,"Cell"]
                
    ax3[i+1].set_title(time,fontsize=30)   
    ax3[i+1].legend(handles=[mlines.Line2D([],[],color=color[j],label=j) for j in data.index.unique(level=0)],fontsize=20)      
    
    #Save separations
    adataAux.write("results/dimensional_reduction_"+time+".h5ad")    
    
adata.write("results/QC.h5ad")
    
#Save figures
fig.savefig("plots/scatterplot_clusters.png",bbox_inches="tight")
fig2.savefig("plots/condition_contribution_clusters.png",bbox_inches="tight")
fig3.savefig("plots/cluster_contribution_conditions.png",bbox_inches="tight")

plt.close(fig)
plt.close(fig2)
plt.close(fig3)

# DE expression

In [33]:
genes = pd.read_excel("Marker_genes_scRNAseq_Gx.xlsx",header=None)

<ipython-input-33-2dff338aca04>:1: FutureWarning: Your version of xlrd is 1.2.0. In xlrd >= 2.0, only the xls format is supported. As a result, the openpyxl engine will be used if it is installed and the engine argument is not specified. Install openpyxl instead.
  genes = pd.read_excel("Marker_genes_scRNAseq_Gx.xlsx",header=None)


In [44]:
if not os.path.isdir("plots/DE_genes_umap"):
    os.mkdir("plots/DE_genes_umap")
    
not_found_genes = []
for gene in genes.values[:,0]:
        
    if gene in adata.var.loc[:,"gene_name"].values:
        fig,ax = plt.subplots(6,1,figsize=[10,40])

        sns.scatterplot(x=adata.obsm[umap][:,0],y=adata.obsm[umap][:,1],hue=np.array(adata[:,adata.var["gene_name"]==gene].X.todense())[:,0],ax=ax[0])
        ax[0].set_title("Global",fontsize=30)  
        
        for i,time in enumerate(np.sort(adata.obs["Time"].unique())):

            adataAux = scp.read("results/dimensional_reduction_"+time+".h5ad")

            sns.scatterplot(x=adataAux.obsm[umap][:,0],y=adataAux.obsm[umap][:,1],hue=np.array(adataAux[:,adataAux.var["gene_name"]==gene].X.todense())[:,0],ax=ax[i+1])
            ax[i+1].set_title(time,fontsize=30)  
            
        fig.savefig("plots/gene_expression_umap/"+gene+".png")
        plt.close(fig)
    else:
        not_found_genes.append(gene)
        
print("Not found genes: ",not_found_genes)

Not found genes:  ['Pax5']


In [42]:
if not os.path.isdir("tables"):
    os.mkdir("tables")
    
scp.tl.rank_genes_groups(adata,groupby="leiden_global",method="wilcoxon",use_raw=False)
n_genes = 200
        
writer = pd.ExcelWriter("tables/DE_global.xlsx", engine='xlsxwriter')
for k,group in enumerate(adata.uns["rank_genes_groups"]["names"].dtype.names):
            
    l = pd.DataFrame(columns=["names","logfoldchanges","pvals","pvals_adj","scores"])
    l.loc[:,"names"] = adata.var.loc[[n[k] for n in adata.uns["rank_genes_groups"]["names"]][:n_genes],"gene_name"].values
    l.loc[:,"logfoldchanges"] = [n[k] for n in adata.uns["rank_genes_groups"]["logfoldchanges"]][:n_genes]
    l.loc[:,"pvals"] = [n[k] for n in adata.uns["rank_genes_groups"]["pvals"]][:n_genes]
    l.loc[:,"pvals_adj"] = [n[k] for n in adata.uns["rank_genes_groups"]["pvals_adj"]][:n_genes]
    l.loc[:,"scores"] = [n[k] for n in adata.uns["rank_genes_groups"]["scores"]][:n_genes]
        
    l.to_excel(writer, sheet_name="cluster_"+str(k))
        
writer.save()

for j,time in enumerate(adata.obs["Time"].unique()[:]):

    adataAux = scp.read("results/dimensional_reduction_"+time+".h5ad")
    
    scp.tl.rank_genes_groups(adataAux,groupby="leiden",method="wilcoxon",use_raw=False)
    n_genes = 200
        
    writer = pd.ExcelWriter("tables/DE_"+time+".xlsx", engine='xlsxwriter')
    for k,group in enumerate(adataAux.uns["rank_genes_groups"]["names"].dtype.names):
            
        l = pd.DataFrame(columns=["names","logfoldchanges","pvals","pvals_adj","scores"])
        l.loc[:,"names"] = adataAux.var.loc[[n[k] for n in adataAux.uns["rank_genes_groups"]["names"]][:n_genes],"gene_name"].values
        l.loc[:,"logfoldchanges"] = [n[k] for n in adataAux.uns["rank_genes_groups"]["logfoldchanges"]][:n_genes]
        l.loc[:,"pvals"] = [n[k] for n in adataAux.uns["rank_genes_groups"]["pvals"]][:n_genes]
        l.loc[:,"pvals_adj"] = [n[k] for n in adataAux.uns["rank_genes_groups"]["pvals_adj"]][:n_genes]
        l.loc[:,"scores"] = [n[k] for n in adataAux.uns["rank_genes_groups"]["scores"]][:n_genes]
        
        l.to_excel(writer, sheet_name="cluster_"+str(k))
        
    writer.save()